In [1]:
import os, sys, torch
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:128'
os.environ["TORCH_NVML_DISABLED"] = "1"
torch.cuda.empty_cache()


os.chdir("/scratch/jq2uw/MME/instruct_vlm_edit")

repo_root = "/scratch/jq2uw/MME/instruct_vlm_edit"
if repo_root not in sys.path:
    sys.path.append(repo_root)

from revlm import *
import argparse


In [2]:
cfg_path = os.path.join(repo_root, "revlm", "config", "config.yaml")

# Simulate CLI overrides
args = argparse.Namespace(
    config=cfg_path,
    editor="ft", 
    split="all",
    task="mc",
    rationale=False,
    dataset_name="aokvqa",
    model_name="qwen3",
    seed=333,
    batch_size=1,
    n_iter=100
)

config = configure_args(args, config_path=cfg_path)
print(config)
ds = VQADataset(config)
df = ds.load_df()


Task evaluation metrics will be saved to results/te/ft/Qwen3-VL-8B-Instruct/aokvqa
Edit evaluation metrics will be saved to results/ee/ft/Qwen3-VL-8B-Instruct/aokvqa
Predictions will be saved to results/pred/Qwen3-VL-8B-Instruct/aokvqa
Unified filename to save: mc_all.json
NestedConfig(batch_size=1, n_iter=100, max_n_edits=5000, seed=333, device='cuda', ckpt_dir=None, dropout=None, task_dir='results/te/ft/Qwen3-VL-8B-Instruct/aokvqa', edit_dir='results/ee/ft/Qwen3-VL-8B-Instruct/aokvqa', pred_dir='results/pred/Qwen3-VL-8B-Instruct/aokvqa', fname='mc_all.json', model=AttrNS(name='Qwen/Qwen3-VL-8B-Instruct', class_name='VQAModel', pt=None, inner_params=['model.visual.deepstack_merger_list.2.linear_fc2.weight'], processor_class=None, tokenizer_class=None, temperature=1.0), editor=AttrNS(_name='ft', edit_lr='1e-4'), experiment=AttrNS(task='mc', dataset_name='aokvqa', pred_by='label_maxprob', split='all', suffix=''))


In [3]:
df[df['uid']==184]

,uid,image_path,question,answer,rationale,choices,idx_choices,image_info_source,image_info_split,image_info_id
183,184,data/images/aokvqa/train2017/000000241617.jpg,That cake is for two people who are involved how?,romantically,These are love birds on the top of the cake us...,rivals; siblings; colleagues; romantically,(A) rivals\n(B) siblings\n(C) colleagues\n(D) ...,COCO,train,train_241617


# Get edit_ds by eval

In [4]:
cfg_path = os.path.join(repo_root, "revlm", "config", "config.yaml")

# Simulate CLI overrides
args = argparse.Namespace(
    config=cfg_path,
    editor="ft", 
    split="all",
    task="mc",
    rationale=False,
    dataset_name="fvqa",
    model_name="qwen3",
    seed=333,
    batch_size=1,
    n_iter=100
)

config = configure_args(args, config_path=cfg_path)
print(config)

# model
model = VQAModel(config)

# dataset
ds = VQADataset(config)
random.seed(getattr(config, "seed", 0))
ds.data = random.sample(ds.data, 50)
ds.set_dataloader(shuffle_choices=True)
ds.task_generate(model)
print(ds.task_engineer.eval(ds))

Task evaluation metrics will be saved to results/te/ft/Qwen3-VL-8B-Instruct/fvqa
Edit evaluation metrics will be saved to results/ee/ft/Qwen3-VL-8B-Instruct/fvqa
Predictions will be saved to results/pred/Qwen3-VL-8B-Instruct/fvqa
Unified filename to save: mc_all.json
NestedConfig(batch_size=1, n_iter=100, max_n_edits=5000, seed=333, device='cuda', ckpt_dir=None, dropout=None, task_dir='results/te/ft/Qwen3-VL-8B-Instruct/fvqa', edit_dir='results/ee/ft/Qwen3-VL-8B-Instruct/fvqa', pred_dir='results/pred/Qwen3-VL-8B-Instruct/fvqa', fname='mc_all.json', model=AttrNS(name='Qwen/Qwen3-VL-8B-Instruct', class_name='VQAModel', pt=None, inner_params=['model.visual.deepstack_merger_list.2.linear_fc2.weight'], processor_class=None, tokenizer_class=None, temperature=1.0), editor=AttrNS(_name='ft', edit_lr='1e-4'), experiment=AttrNS(task='mc', dataset_name='fvqa', pred_by='label_maxprob', split='all', suffix=''))
{'uid': 4546, 'image': 'data/images/fvqa/COCO_val2014_000000014549.jpg', 'question': 'Wh

In [5]:
pred_res_dir = os.path.join("results", "test", "pred", f"{config.model.name}", f"{config.experiment.dataset_name}")
os.makedirs(pred_res_dir, exist_ok=True)
pred_out_path = os.path.join(pred_res_dir, f"{config.experiment.task}_{config.experiment.split}.json")
edit_ds = ds.get_edits()
edit_ds.snap(out_path=pred_out_path)


# edit

In [6]:
# model = VQAModel(config)
# load the prediction set back
import json
pred_set = json.load(open(pred_out_path))
edit_ds = VQADataset(config)
edit_ds.data = pred_set
edit_ds = edit_ds.get_edits()
print(len(edit_ds.data))

2


In [7]:
import copy
model_old = copy.deepcopy(model)
# model_old_weights = copy.deepcopy(model.model.state_dict())
edit_ds.data

[{'uid': 4404,
  'image': 'data/images/fvqa/COCO_val2014_000000105960.jpg',
  'question': 'What can likely be found in this place?',
  'answer': 'boat',
  'rationale': 'You are likely to find boat in a beach.',
  'choices': 'tree; car; boat; sandwich',
  'idx_choices': '(A) tree\n(B) car\n(C) boat\n(D) sandwich',
  'idx': 0,
  'gold': {'label': 'boat',
   'choices': {'str': 'tree; car; boat; sandwich',
    'ls': ['tree', 'car', 'boat', 'sandwich']},
   'label_train': 'boat'},
  'prompt': 'Choose the correct answer from the options. What can likely be found in this place? Options: tree; car; boat; sandwich',
  'pred': {'answer': 'tree',
   'label_text': 'tree',
   'label_scores': {'sandwich': {'avg_nll': 5.36581563949585,
     'sum_nll': 10.7316312789917,
     'num_tokens': 2,
     'prob': 2.184707292192121e-05},
    'car': {'avg_nll': 3.606623649597168,
     'sum_nll': 3.606623649597168,
     'num_tokens': 1,
     'prob': 0.027148432247252618},
    'tree': {'avg_nll': 0.106623619794845

In [8]:
# minimal single-batch finetune step (ft editor, no history)
editor = get_editor(config, model)
editor.generate = model.model.generate if hasattr(model, 'model') else model.generate
model.model.train()

batch = next(iter(edit_ds.loader))
tokens = model.prepare_training_batch(batch)
editor.edit(config, tokens, batch_history=None)

del tokens
torch.cuda.empty_cache()
# model_new_weights = copy.deepcopy(model.model.state_dict())
model_new = model

Finetuning module model.visual.deepstack_merger_list.2.linear_fc2


In [9]:
edit_ds.task_generate(model_new)
edit_ds.data

[{'uid': 4404,
  'image': 'data/images/fvqa/COCO_val2014_000000105960.jpg',
  'question': 'What can likely be found in this place?',
  'answer': 'boat',
  'rationale': 'You are likely to find boat in a beach.',
  'choices': 'tree; car; boat; sandwich',
  'idx_choices': '(A) tree\n(B) car\n(C) boat\n(D) sandwich',
  'idx': 0,
  'gold': {'label': 'boat',
   'choices': {'str': 'tree; car; boat; sandwich',
    'ls': ['tree', 'car', 'boat', 'sandwich']},
   'label_train': 'boat'},
  'prompt': 'Choose the correct answer from the options. What can likely be found in this place? Options: tree; car; boat; sandwich',
  'pred': {'answer': 'boat',
   'label_text': 'boat',
   'label_scores': {'tree': {'avg_nll': 12.25004768371582,
     'sum_nll': 12.25004768371582,
     'num_tokens': 1,
     'prob': 4.7850375249636195e-06},
    'car': {'avg_nll': 11.37504768371582,
     'sum_nll': 11.37504768371582,
     'num_tokens': 1,
     'prob': 1.1478708299340697e-05},
    'boat': {'avg_nll': 4.73249674541875

In [10]:
# model_old = copy.deepcopy(model)
# model_old.model.load_state_dict(model_old_weights)
edit_ds.task_generate(model_old)
edit_ds.data

[{'uid': 4404,
  'image': 'data/images/fvqa/COCO_val2014_000000105960.jpg',
  'question': 'What can likely be found in this place?',
  'answer': 'boat',
  'rationale': 'You are likely to find boat in a beach.',
  'choices': 'tree; car; boat; sandwich',
  'idx_choices': '(A) tree\n(B) car\n(C) boat\n(D) sandwich',
  'idx': 0,
  'gold': {'label': 'boat',
   'choices': {'str': 'tree; car; boat; sandwich',
    'ls': ['tree', 'car', 'boat', 'sandwich']},
   'label_train': 'boat'},
  'prompt': 'Choose the correct answer from the options. What can likely be found in this place? Options: tree; car; boat; sandwich',
  'pred': {'answer': 'tree',
   'label_text': 'tree',
   'label_scores': {'tree': {'avg_nll': 0.09182452410459518,
     'sum_nll': 0.09182452410459518,
     'num_tokens': 1,
     'prob': 0.912401326387312},
    'car': {'avg_nll': 3.841824531555176,
     'sum_nll': 3.841824531555176,
     'num_tokens': 1,
     'prob': 0.021457622352790678},
    'boat': {'avg_nll': 2.716824531555176,


## eval edits

### reliability

In [11]:
from revlm.metrics import *

In [12]:
reliability(model_old, edit_ds)

0.0

In [13]:
reliability(model_new, edit_ds)

0.5

### generality

In [14]:

# related_texts=
# related_images=
# unrelated_texts=
# unrelated_images=

related_texts={}
related_images={}
for ex in edit_ds.data:
    print(ex)
    related_texts[ex['uid']] = [ex['question'], ex['question'], ex['question'], ex['question']]
    related_images[ex['uid']] = [ex['image'], ex['image'], ex['image'], ex['image']]

print(image_generality(model_new, edit_ds, related_images))
print(text_generality(model_new, edit_ds, related_texts))



{'uid': 4404, 'image': 'data/images/fvqa/COCO_val2014_000000105960.jpg', 'question': 'What can likely be found in this place?', 'answer': 'boat', 'rationale': 'You are likely to find boat in a beach.', 'choices': 'tree; car; boat; sandwich', 'idx_choices': '(A) tree\n(B) car\n(C) boat\n(D) sandwich', 'idx': 0, 'gold': {'label': 'boat', 'choices': {'str': 'tree; car; boat; sandwich', 'ls': ['tree', 'car', 'boat', 'sandwich']}, 'label_train': 'boat'}, 'prompt': 'Choose the correct answer from the options. What can likely be found in this place? Options: tree; car; boat; sandwich', 'pred': {'answer': 'boat', 'label_text': 'boat', 'label_scores': {'tree': {'avg_nll': 12.25004768371582, 'sum_nll': 12.25004768371582, 'num_tokens': 1, 'prob': 4.7850375249636195e-06}, 'car': {'avg_nll': 11.37504768371582, 'sum_nll': 11.37504768371582, 'num_tokens': 1, 'prob': 1.1478708299340697e-05}, 'boat': {'avg_nll': 4.732496745418757e-05, 'sum_nll': 4.732496745418757e-05, 'num_tokens': 1, 'prob': 0.9999836

In [ ]:
print(locality(model_old, model_new, edit_ds, sample_size=100))


In [ ]:
# # 20m30s
# editeval(model_old = model_old,
#         model_new = model_new,
#         edit_ds = edit_ds,
#         related_texts = related_texts,
#         related_images = related_images, 
#         loc_sample_size = 1000)

In [ ]:
# {'reliability': 0.5,
#  'text_generality': 0.5,
#  'image_generality': 0.5,
#  'locality': 0.84,
#  'combined': 1.8399999999999999}

In [ ]:
# Test rationale_generality on a tiny dummy edit set
import copy
from revlm.metrics import rationale_generality

# Require existing model/config/edit_ds from earlier cells
try:
    model  # noqa: F401
    config  # noqa: F401
    edit_ds  # noqa: F401
except NameError:
    raise RuntimeError("Please run the earlier cells to define model, config, and edit_ds before this test.")

# Build a 2-sample dummy edit set from current edit_ds
_dummy = copy.deepcopy(edit_ds)
_dummy.data = _dummy.data[:2]
_dummy.set_dataloader(shuffle_choices=False)

# Map each uid to the other's uid to form a simple related_rationale
uids = [_dummy.data[i]["uid"] for i in range(len(_dummy.data))]
related_rationale = {}
if len(uids) >= 2:
    related_rationale = {uids[0]: [uids[1]], uids[1]: [uids[0]]}
else:
    # If only one example exists, just point to itself (degenerate case)
    related_rationale = {uids[0]: [uids[0]]}

print("rationale_generality:", rationale_generality(model, _dummy, related_rationale))


rationale_generality: 0.5


In [ ]:
# Test edit1_generality on the same tiny dummy edit set
import copy
from revlm.editors import get_editor
from revlm.metrics import edit1_generality

# Require existing model/config/edit_ds from earlier cells
try:
    model  # noqa: F401
    config  # noqa: F401
    edit_ds  # noqa: F401
except NameError:
    raise RuntimeError("Please run the earlier cells to define model, config, and edit_ds before this test.")

# Build dummy edit set (reuse 2 examples)
_dummy = copy.deepcopy(edit_ds)
_dummy.data = _dummy.data[:2]
# Keep evaluation deterministic and light
_dummy.config.n_iter = 1  # single training step inside editor.edit
_dummy.set_dataloader(shuffle_choices=True)

# Fresh base model for editing
model_old = copy.deepcopy(model)
editor = get_editor(config, model_old)
editor.generate = model_old.model.generate if hasattr(model_old, 'model') else model_old.generate

print("edit1_generality:", edit1_generality(model_old, _dummy, editor))


Finetuning model.visual.deepstack_merger_list.2.linear_fc2.weight
edit1_generality: 0.5
